In [ ]:
%pip install ucimlrepo

import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import numpy as np
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import f1_score
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance

credit_default = fetch_ucirepo(id=350)

X = credit_default.data.features.copy()
y = credit_default.data.targets.copy()

print(f'Predictor table: {X.shape[0]:,} rows x {X.shape[1]} columns')
print(f'Outcome table:   {y.shape[0]:,} rows x {y.shape[1]} column')

display(X.head())
display(y.head())

variable_dictionary = credit_default.variables
display(variable_dictionary)

y.columns = ["default_next_month"]

feature_names = [
    "LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE",
    "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6",
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3",
    "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3",
    "PAY_AMT4", "PAY_AMT5", "PAY_AMT6",
]

X.columns = feature_names
y.columns = ["default_next_month"]

df = pd.concat([X, y], axis=1)

print('Final working-table shape:', df.shape)
print('Missing cells:', df.isna().sum().sum())
print('\nOutcome counts:')
print(df['default_next_month'].value_counts())
print(f"\nDefault rate: {df['default_next_month'].mean():.2%}")

X = df.drop(columns="default_next_month")
y = df["default_next_month"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training observations:", len(X_train))
print("Test observations:", len(X_test))

print(f"\nTraining default rate: {y_train.mean():.2%}")
print(f"Test default rate:     {y_test.mean():.2%}")

baseline_predictions = pd.Series(
    0,
    index=y_test.index,
    name="baseline_prediction"
)

print("Accuracy:", accuracy_score(y_test, baseline_predictions))
print("Precision:", precision_score(y_test, baseline_predictions, zero_division=0))
print("Recall:", recall_score(y_test, baseline_predictions, zero_division=0))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, baseline_predictions))

categorical_features = ["SEX", "EDUCATION", "MARRIAGE"]

numeric_features = [
    column for column in X_train.columns
    if column not in categorical_features
]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=2000, random_state=42)),
    ]
)

logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_test)
logistic_probabilities = logistic_model.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, logistic_predictions), 4))
print(
    "Precision:",
    round(precision_score(y_test, logistic_predictions, zero_division=0), 4),
)
print("Recall:", round(recall_score(y_test, logistic_predictions, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, logistic_probabilities), 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, logistic_predictions))

tree_model = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=100,
    random_state=42
)

tree_model.fit(X_train, y_train)

tree_predictions = tree_model.predict(X_test)
tree_probabilities = tree_model.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, tree_predictions), 4))
print(
    "Precision:",
    round(precision_score(y_test, tree_predictions, zero_division=0), 4),
)
print("Recall:", round(recall_score(y_test, tree_predictions, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, tree_probabilities), 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, tree_predictions))

plt.figure(figsize=(20, 10))

plot_tree(
    tree_model,
    feature_names=X_train.columns,
    class_names=["No default", "Default"],
    filled=True,
    rounded=True,
    proportion=True,
    fontsize=9
)

plt.title("Initial Decision Tree for Next-Month Default")
plt.show()

train_probabilities = tree_model.predict_proba(X_train)[:, 1]
test_probabilities = tree_model.predict_proba(X_test)[:, 1]

print(
    "Training ROC-AUC:",
    round(roc_auc_score(y_train, train_probabilities), 4)
)

print(
    "Test ROC-AUC:",
    round(roc_auc_score(y_test, test_probabilities), 4)
)

print("Number of leaves:", tree_model.get_n_leaves())
print("Tree depth:", tree_model.get_depth())

deep_tree = DecisionTreeClassifier(
    min_samples_leaf=1,
    random_state=42
)

deep_tree.fit(X_train, y_train)

deep_train_auc = roc_auc_score(
    y_train,
    deep_tree.predict_proba(X_train)[:, 1]
)

deep_test_auc = roc_auc_score(
    y_test,
    deep_tree.predict_proba(X_test)[:, 1]
)

print("Training ROC-AUC:", round(deep_train_auc, 4))
print("Test ROC-AUC:", round(deep_test_auc, 4))
print("Number of leaves:", deep_tree.get_n_leaves())
print("Tree depth:", deep_tree.get_depth())

unpruned_tree = DecisionTreeClassifier(
    random_state=42
)
\
unpruned_tree.fit(X_train, y_train)

pruning_path = unpruned_tree.cost_complexity_pruning_path(
    X_train,
    y_train
)

candidate_alphas = pruning_path.ccp_alphas[:-1]

print("Number of candidate pruning strengths:", len(candidate_alphas))
print("First five alpha values:", candidate_alphas[:5])
print("Last five alpha values:", candidate_alphas[-5:])

# Select 50 representative pruning strengths from the full path.
candidate_indices = np.linspace(
    0,
    len(candidate_alphas) - 1,
    num=50,
    dtype=int
)

candidate_alphas_small = np.unique(
    candidate_alphas[candidate_indices]
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

pruning_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid={"ccp_alpha": candidate_alphas_small},
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

pruning_search.fit(X_train, y_train)

print("Best ccp_alpha:", pruning_search.best_params_["ccp_alpha"])
print("Best cross-validation ROC-AUC:", round(pruning_search.best_score_, 4))

pruned_tree = pruning_search.best_estimator_

print("Pruned-tree leaves:", pruned_tree.get_n_leaves())
print("Pruned-tree depth:", pruned_tree.get_depth())

pruned_predictions = pruned_tree.predict(X_test)
pruned_probabilities = pruned_tree.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, pruned_predictions), 4))
print(
    "Precision:",
    round(precision_score(y_test, pruned_predictions, zero_division=0), 4),
)
print("Recall:", round(recall_score(y_test, pruned_predictions, zero_division=0), 4))
print("Test ROC-AUC:", round(roc_auc_score(y_test, pruned_probabilities), 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, pruned_predictions))

pruned_train_auc = roc_auc_score(
    y_train,
    pruned_tree.predict_proba(X_train)[:, 1]
)

pruned_test_auc = roc_auc_score(
    y_test,
    pruned_probabilities
)

print("Training ROC-AUC:", round(pruned_train_auc, 4))
print("Test ROC-AUC:", round(pruned_test_auc, 4))
print("ROC-AUC gap:", round(pruned_train_auc - pruned_test_auc, 4))

random_forest = RandomForestClassifier(
    n_estimators=500,
    max_features="sqrt",
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

random_forest.fit(X_train, y_train)

rf_predictions = random_forest.predict(X_test)
rf_probabilities = random_forest.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, rf_predictions), 4))
print(
    "Precision:",
    round(precision_score(y_test, rf_predictions, zero_division=0), 4),
)
print("Recall:", round(recall_score(y_test, rf_predictions, zero_division=0), 4))
print("Test ROC-AUC:", round(roc_auc_score(y_test, rf_probabilities), 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, rf_predictions))

random_forest = RandomForestClassifier(
    n_estimators=500,
    max_features="sqrt",
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

random_forest.fit(X_train, y_train)

rf_predictions = random_forest.predict(X_test)
rf_probabilities = random_forest.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, rf_predictions), 4))
print(
    "Precision:",
    round(precision_score(y_test, rf_predictions, zero_division=0), 4),
)
print("Recall:", round(recall_score(y_test, rf_predictions, zero_division=0), 4))
print("Test ROC-AUC:", round(roc_auc_score(y_test, rf_probabilities), 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, rf_predictions))

feature_importance = pd.Series(
    random_forest.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

display(feature_importance.head(10))

feature_importance.head(10).sort_values().plot(
    kind="barh",
    figsize=(8, 5),
    title="Random Forest: Top 10 Feature Importances"
)

plt.xlabel("Relative impurity-based importance")
plt.show()

boosting_model = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=2,
    min_samples_leaf=20,
    random_state=42
)

boosting_model.fit(X_train, y_train)

boosting_predictions = boosting_model.predict(X_test)
boosting_probabilities = boosting_model.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, boosting_predictions), 4))
print(
    "Precision:",
    round(precision_score(y_test, boosting_predictions, zero_division=0), 4),
)
print("Recall:", round(recall_score(y_test, boosting_predictions, zero_division=0), 4))
print("Test ROC-AUC:", round(roc_auc_score(y_test, boosting_probabilities), 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, boosting_predictions))

model_results = pd.DataFrame(
    {
        "Model": [
            "All-no-default baseline",
            "Small decision tree",
            "Cross-validated pruned tree",
            "Random forest",
            "Gradient boosting",
        ],
        "Accuracy": [
            0.7788,
            0.8183,
            0.8179,
            0.8171,
            0.8195,
        ],
        "Precision": [
            0.0,
            0.6616,
            0.6626,
            0.6579,
            0.6704,
        ],
        "Recall": [
            0.0,
            0.3653,
            0.3599,
            0.3605,
            0.3617,
        ],
        "Test ROC-AUC": [
            0.5,
            0.7308,
            0.7614,
            0.7741,
            0.7763,
        ],
    }
)

display(model_results.style.format({
    "Accuracy": "{:.2%}",
    "Precision": "{:.2%}",
    "Recall": "{:.2%}",
    "Test ROC-AUC": "{:.4f}",
    }))

threshold_results = []

for threshold in [0.20, 0.30, 0.40, 0.50, 0.60]:
    threshold_predictions = (
        boosting_probabilities >= threshold
    ).astype(int)

    threshold_results.append(
        {
            "Threshold": threshold,
            "Precision": precision_score(
                y_test, threshold_predictions, zero_division=0
            ),
            "Recall": recall_score(
                y_test, threshold_predictions, zero_division=0
            ),
            "F1 score": f1_score(
                y_test, threshold_predictions, zero_division=0
            ),
            "Accounts flagged": threshold_predictions.sum(),
        }
    )

threshold_table = pd.DataFrame(threshold_results)

display(
    threshold_table.style.format(
        {
            "Threshold": "{:.2f}",
            "Precision": "{:.2%}",
            "Recall": "{:.2%}",
            "F1 score": "{:.4f}",
        }
    )
)
threshold_table.plot(
    x="Threshold",
    y=["Precision", "Recall"],
    marker="o",
    figsize=(8, 5),
    ylim=(0, 1),
    title="Gradient Boosting: Precision-Recall Trade-off by Threshold",
)

plt.ylabel("Score")
plt.show()

observed_default_rate, predicted_probability = calibration_curve(
    y_test,
    boosting_probabilities,
    n_bins=10,
    strategy="quantile"
)

plt.figure(figsize=(7, 6))

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="gray",
    label="Perfect calibration",
)

plt.plot(
    predicted_probability,
    observed_default_rate,
    marker="o",
    label="Gradient boosting",
)

plt.xlabel("Mean predicted probability")
plt.ylabel("Observed default frequency")
plt.title("Calibration Curve: Gradient-Boosting Model")
plt.legend()
plt.show()

calibration_table = pd.DataFrame(
    {
        "Mean predicted probability": predicted_probability,
        "Observed default rate": observed_default_rate,
    }
)

display(
    calibration_table.style.format(
        {
            "Mean predicted probability": "{:.2%}",
            "Observed default rate": "{:.2%}",
        }
    )
)
split_results = []

for seed in [1, 7, 21, 42, 99]:
    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=seed,
        stratify=y,
    )

    model = GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=2,
        min_samples_leaf=20,
        random_state=42,
    )

    model.fit(X_train_s, y_train_s)

    probabilities = model.predict_proba(X_test_s)[:, 1]

    split_results.append(
        {
            "Split seed": seed,
            "Test ROC-AUC": roc_auc_score(y_test_s, probabilities),
        }
    )

split_stability = pd.DataFrame(split_results)

display(
    split_stability.style.format(
        {"Test ROC-AUC": "{:.4f}"}
    )
)

print(
    "Mean test ROC-AUC:",
    round(split_stability["Test ROC-AUC"].mean(), 4)
)

print(
    "Standard deviation:",
    round(split_stability["Test ROC-AUC"].std(), 4)
)
permutation_results = permutation_importance(
    boosting_model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
)

permutation_table = pd.DataFrame(
    {
        "Feature": X_test.columns,
        "Mean ROC-AUC decrease": permutation_results.importances_mean,
        "Standard deviation": permutation_results.importances_std,
    }
).sort_values(
    "Mean ROC-AUC decrease",
    ascending=False,
)

display(
    permutation_table.head(10).style.format(
        {
            "Mean ROC-AUC decrease": "{:.4f}",
            "Standard deviation": "{:.4f}",
        }
    )
)

permutation_table.head(10).sort_values(
    "Mean ROC-AUC decrease"
).plot(
    x="Feature",
    y="Mean ROC-AUC decrease",
    kind="barh",
    figsize=(8, 5),
    title="Gradient Boosting: Permutation Importance",
    legend=False,
)

plt.xlabel("Test ROC-AUC decrease after feature shuffling")
plt.show()

subgroup_results = []

for group_name, mask in {
    "PAY_0 < 2": X_test["PAY_0"] < 2,
    "PAY_0 >= 2": X_test["PAY_0"] >= 2,
}.items():
    group_y = y_test[mask]
    group_probabilities = boosting_probabilities[mask]

    subgroup_results.append(
        {
            "Group": group_name,
            "Accounts": mask.sum(),
            "Observed default rate": group_y.mean(),
            "ROC-AUC": roc_auc_score(group_y, group_probabilities),
        }
    )

subgroup_table = pd.DataFrame(subgroup_results)

display(
    subgroup_table.style.format(
        {
            "Observed default rate": "{:.2%}",
            "ROC-AUC": "{:.4f}",
        }
    )
)
operating_threshold = 0.30

subgroup_threshold_results = []

for group_name, mask in {
    "PAY_0 < 2": X_test["PAY_0"] < 2,
    "PAY_0 >= 2": X_test["PAY_0"] >= 2,
}.items():
    group_y = y_test[mask]
    group_predictions = (
        boosting_probabilities[mask] >= operating_threshold
    ).astype(int)

    subgroup_threshold_results.append(
        {
            "Group": group_name,
            "Accounts": mask.sum(),
            "Precision": precision_score(
                group_y, group_predictions, zero_division=0
            ),
            "Recall": recall_score(
                group_y, group_predictions, zero_division=0
            ),
            "Accounts flagged": group_predictions.sum(),
        }
    )

subgroup_threshold_table = pd.DataFrame(subgroup_threshold_results)

display(
    subgroup_threshold_table.style.format(
        {
            "Precision": "{:.2%}",
            "Recall": "{:.2%}",
        }
    )
)

In [ ]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

credit_default = fetch_ucirepo(id=350)

X_raw = credit_default.data.features.copy()
y_raw = credit_default.data.targets.copy()

print("Feature-table shape:", X_raw.shape)
print("Target-table shape:", y_raw.shape)
print("\nOriginal feature names:")
print(X_raw.columns.tolist())

print("\nTarget counts:")
print(y_raw.iloc[:, 0].value_counts().sort_index())

assert X_raw.shape == (30000, 23)
assert y_raw.shape == (30000, 1)
assert X_raw.index.equals(y_raw.index)
assert set(y_raw.iloc[:, 0].unique()) == {0, 1}

print("\nAll basic source-data checks passed.")
display(
    credit_default.variables[
        ["name", "role", "description"]
    ]
)